In [1]:
import boto3
import botocore
from botocore import UNSIGNED
from botocore.config import Config
from itertools import product
from botocore.config import Config

import rasterio
import numpy as np
import cv2
import matplotlib.pyplot as plt

Steps -

Target Label Generation Data Selection -

Find images within a certain data range (eventually depends on when the CDL data is taken) and ensure we are getting clear images. Because we can overlay the CDL data onto a couple of images, if a field is a field in a couple of images then it is definitely a field is the assumption. The metrics we are checking are in the S2B_1CCV_20190909_1_L2A.json (example) file. They are cloud cover, no data percentage, and sun elevation for now. The challenge is - cannot find clear image for every part of the US on the exact same date, even same season. If the season varies, the crop field is inconsistent. The field detection will also be inaccurate.


So, we will be using one source for target label generation with simply the existing edge detection algorithm. In this case, we are sure we only have one source of truth, and we can ensure the accuracy!



For actual modeling, we only need to select the US continent in the sentinel-2! With some references we nailed down the tiles and the latitude.


In [ ]:
# Set up parameters
bucket_name = "sentinel-cogs"
base_prefix = "sentinel-s2-l2a-cogs"
# Initialize S3 client
# Initialize S3 client with unsigned config
config = Config(
    signature_version=botocore.UNSIGNED,
    retries = dict(
        max_attempts = 3
    )
)
s3 = boto3.client('s3', config=config)

# Generate all possible tiles for US
# Because the CDL map does not include alaska or hawaii, we will only consider UTM zones 
# and latitude bands that cover the continental US
# UTM zones for US: Primarily 10-19 (Continental US)
# Latitude bands: Primarily R, S, T, U (Continental US)
# Reference: 
# https://www.usgs.gov/media/images/mapping-utm-grid-conterminous-48-united-states#:~:text=The%20Universal%20Transverse%20Mercator%20grid,Zone%2019%20in%20New%20England.
# https://earth-info.nga.mil/index.php?dir=coordsys&action=coordsys
utm_zones = range(10, 19)  # 1-60
latitude_bands = 'UTSR'  # Sentinel-2 latitude bands

# Discovery the grid squares
grid_squares_found = set()

# s3://sentinel-cogs/sentinel-s2-l2a-cogs/7/R/
# 7 is UTM, R is latitude band for example
for utm in utm_zones:
    print(f"\nChecking UTM zone: {utm}")
    
    for lat_band in latitude_bands:
        print(f"Checking latitude band: {lat_band}")
        
        # List initial contents
        prefix = f"{base_prefix}/{utm}/{lat_band}/"
        print(prefix)
        response = s3.list_objects_v2(
            Bucket=bucket_name,
            Prefix=prefix,
            Delimiter='/'
        )
        print(response)
        
        # Process each grid square prefix
        for prefix in response.get('CommonPrefixes', []):
            grid_square = prefix.get('Prefix').split('/')[3]
            grid_squares_found.add(grid_square)
                

print(f"Found {len(grid_squares_found)} grid squares:")
print(', '.join(sorted(grid_squares_found)))

Edge Detection Algorithm - Canny edge detection

Reference: https://learnopencv.com/edge-detection-using-opencv/#canny-edge

1. Noise reduction - a Gaussian blur filter is used to essentially remove or minimize unnecessary detail that could lead to undesirable edges.

2. Apply non-maximum suppression of edges to filter out unwanted pixels (which may not actually constitute an edge). If the gradient magnitude of the current pixel is greater than its neighboring pixels, it is left unchanged. Otherwise, the magnitude of the current pixel is set to zero.

3. The gradient magnitudes are compared with two threshold values, one smaller than the other. If the gradient magnitude value is higher than the larger threshold value, those pixels are associated with solid edges and are included in the final edge map. If the gradient magnitude values are lower than the smaller threshold value, the pixels are suppressed and excluded from the final edge map. All the other pixels, whose gradient magnitudes fall between these two thresholds, are marked as ‘weak’ edges (i.e. they become candidates for being included in the final edge map). If the ‘weak’ pixels are connected to those associated with solid edges, they are also included in the final edge map. 




In [ ]:
def detect_field_edges(input_tif, output_tif):

    with rasterio.open(input_tif) as src:
        # Read first band because the file is single band
        cdl_data = src.read(1) 
        meta = src.meta.copy()
        
        # Apply Canny edge detection

        # Convert to uint8 for OpenCV with min max normalization to 0-255 (8bit image)
        cdl_scaled = ((cdl_data - cdl_data.min()) * (255.0 / (cdl_data.max() - 
                                                              cdl_data.min()))).astype(np.uint8)
        
        # Apply Gaussian blur to reduce noise
        # 5x5 pixel window will be used for blurring, the larger the more blurring it creates
        blurred = cv2.GaussianBlur(cdl_scaled, (5, 5), 0)
        
        # lower values will detect more edges (more sensitive) but may include more noise
        edges = cv2.Canny(blurred, threshold1=30, threshold2=100)
        
        # Dilate (thicken) edges to connect broken lines, help connect nearby edges that might have gaps
        # 3x3 matrix
        kernel = np.ones((3,3), np.uint8)
        # the more iterations, the thicker the edges
        dilated_edges = cv2.dilate(edges, kernel, iterations=1)
        
        # Find contours in the edge image
        # RETR_EXTERNAL only retrieves the outer/external contours, ignores contours inside other contours
        contours, _ = cv2.findContours(dilated_edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        
        # Create mask for fields, this creates blank canvas for the field mask
        field_mask = np.zeros_like(cdl_data, dtype=np.uint8)
        
        # Fill contours to create field regions
        # -1 means draw all contours (could use index for specific contour)
        # 1 is the value to fill with (makes fields = 1, background = 0)
        # thickness=cv2.FILLED fills the entire contour area
        cv2.drawContours(field_mask, contours, -1, 1, thickness=cv2.FILLED)
        
        meta.update({
            'dtype': 'uint8',
            'nodata': 0 # Specify that 0 represents no data/background
        })
        
        # Save the result
        with rasterio.open(output_tif, 'w', **meta) as dst:
            dst.write(field_mask, 1)

        # Create and save visualization
        viz_output = output_tif.replace('.tif', '_viz.png')
        
        # Create a visualization with different stages
        fig, axes = plt.subplots(2, 2, figsize=(12, 12))
        
        # Original scaled data
        axes[0, 0].imshow(cdl_scaled, cmap='gray')
        axes[0, 0].set_title('Original Scaled Data')
        
        # Edge detection
        axes[0, 1].imshow(edges, cmap='gray')
        axes[0, 1].set_title('Edge Detection')
        
        # Dilated edges
        axes[1, 0].imshow(dilated_edges, cmap='gray')
        axes[1, 0].set_title('Dilated Edges')
        
        # Final field mask
        axes[1, 1].imshow(field_mask, cmap='gray')
        axes[1, 1].set_title('Final Field Mask')
        
        # Remove axes for cleaner look
        for ax in axes.flat:
            ax.axis('off')
        
        plt.tight_layout()
        plt.savefig(viz_output, dpi=300, bbox_inches='tight')
        plt.close()
        
        print(f"Processing complete!")
        print(f"Binary mask saved to: {output_tif}")
        print(f"Visualization saved to: {viz_output}")
        
        return field_mask

# File paths
input_file = "2021_30m_cdls.tif"
output_file = "field_boundaries.tif"

# Run the edge detection
result = detect_field_edges(input_file, output_file)